<a href="https://colab.research.google.com/github/vsai4607/MLA0204/blob/main/program3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import math
import io

class DecisionTreeNode:
    def __init__(self, attribute=None, label=None, branches=None):
        self.attribute = attribute
        self.label = label
        self.branches = branches if branches is not None else {}


def entropy(data):
    target = data["target"]

    if len(target) == 0:
        return 0

    unique, counts = np.unique(target, return_counts=True)

    ent = 0

    for count in counts:
        p = count / len(target)
        ent -= p * math.log2(p)

    return ent


def information_gain(data, attribute):
    total_entropy = entropy(data)
    weighted_entropy = 0

    for value in data[attribute].unique():
        subset = data[data[attribute] == value]

        weighted_entropy += (
            len(subset) / len(data)
        ) * entropy(subset)

    return total_entropy - weighted_entropy


def id3(data, attributes):

    target = data["target"]

    # All examples have same target
    if len(target.unique()) == 1:
        return DecisionTreeNode(label=target.iloc[0])

    # No attributes left
    if len(attributes) == 0:
        return DecisionTreeNode(
            label=target.value_counts().idxmax()
        )

    # Calculate information gain
    gains = {}

    for attribute in attributes:
        gains[attribute] = information_gain(
            data, attribute
        )

    # Select best attribute
    best_attribute = max(
        gains,
        key=gains.get
    )

    node = DecisionTreeNode(
        attribute=best_attribute
    )

    # Create branches
    for value in data[best_attribute].unique():

        subset = data[
            data[best_attribute] == value
        ].drop(columns=[best_attribute])

        new_attributes = attributes.copy()
        new_attributes.remove(best_attribute)

        if len(subset) == 0:

            node.branches[value] = DecisionTreeNode(
                label=target.value_counts().idxmax()
            )

        else:

            node.branches[value] = id3(
                subset,
                new_attributes
            )

    return node


# Function to print decision tree
def print_tree(tree, level=0):

    if tree.label is not None:
        print("    " * level + "-> " + str(tree.label))
        return

    print("    " * level + "[" + str(tree.attribute) + "]")

    for value, branch in tree.branches.items():

        print(
            "    " * (level + 1) +
            str(value) + ":"
        )

        print_tree(
            branch,
            level + 2
        )


# Function to classify a new sample
def classify(sample, tree):

    if tree.label is not None:
        return tree.label

    attribute = tree.attribute
    value = sample[attribute]

    if value in tree.branches:

        return classify(
            sample,
            tree.branches[value]
        )

    # Unknown value
    return "Unknown"


# Play Tennis dataset definition
csv_data = """day,outlook,temp,humidity,wind,play
D1,Sunny,Hot,High,Weak,No
D2,Sunny,Hot,High,Strong,No
D3,Overcast,Hot,High,Weak,Yes
D4,Rain,Mild,High,Weak,Yes
D5,Rain,Cool,Normal,Weak,Yes
D6,Rain,Cool,Normal,Strong,No
D7,Overcast,Cool,Normal,Strong,Yes
D8,Sunny,Mild,High,Weak,No
D9,Sunny,Cool,Normal,Weak,Yes
D10,Rain,Mild,Normal,Weak,Yes
D11,Sunny,Mild,Normal,Strong,Yes
D12,Overcast,Mild,High,Strong,Yes
D13,Overcast,Hot,Normal,Weak,Yes
D14,Rain,Mild,High,Strong,No"""

# Read dataset
data = pd.read_csv(io.StringIO(csv_data))

# Drop non-predictive id column 'day'
if 'day' in data.columns:
    data = data.drop(columns=['day'])

# Rename last column as target
data = data.rename(
    columns={
        data.columns[-1]: "target"
    }
)

# Get attributes
attributes = data.columns[:-1].tolist()


# Build decision tree
root = id3(
    data,
    attributes
)


# Display decision tree
print("\nDECISION TREE")
print("=============")

print_tree(root)


# Test sample
sample = {
    "outlook": "Sunny",
    "temp": "Cool",
    "humidity": "High",
    "wind": "Strong"
}


# Prediction
result = classify(
    sample,
    root
)

print("\nPrediction:", result)


DECISION TREE
[outlook]
    Sunny:
        [humidity]
            High:
                -> No
            Normal:
                -> Yes
    Overcast:
        -> Yes
    Rain:
        [wind]
            Weak:
                -> Yes
            Strong:
                -> No

Prediction: No
